In [ ]:
from pathlib import Path
import json
import re
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, energy_distance, wasserstein_distance, gaussian_kde
from scipy.spatial.distance import jensenshannon

ROOT = Path.cwd()
RESULTS_ROOT = ROOT / "number_extracted_realworld" / "realworld_human"
GT_FILE = ROOT / "ground_truth_values" / "realworld_human" / "ground_truth_values.json"
MAPPING_FILE = ROOT / "realworld_mapping.json"

USE_TEMP_1_RUN_1_100 = False

print("ROOT:", ROOT)
print("RESULTS_ROOT exists:", RESULTS_ROOT.exists())
print("GT_FILE exists:", GT_FILE.exists())
print("MAPPING_FILE exists:", MAPPING_FILE.exists())

In [ ]:
def lehmer_code(perm: np.ndarray) -> np.ndarray:
    """
    Lehmer code L_i = #{ j>i : perm[j] < perm[i] }.
    perm: shape (n,)
    returns: shape (n,) with last digit always 0.
    """
    perm = np.asarray(perm)
    n = perm.shape[0]
    L = np.zeros(n, dtype=int)
    for i in range(n):
        if i + 1 < n:
            L[i] = int(np.sum(perm[i + 1 :] < perm[i]))
        else:
            L[i] = 0
    return L


def lehmer_encode_batch(perms: np.ndarray) -> np.ndarray:
    """
    perms: (m, n)
    returns: (m, n) Lehmer codes
    """
    perms = np.asarray(perms)
    if perms.ndim != 2:
        raise ValueError("perms must be a 2D array of shape (m, n)")
    m, _ = perms.shape
    return np.vstack([lehmer_code(perms[i]) for i in range(m)])


def ks_score(a, b):
    """Calculate Kolmogorov-Smirnov test."""
    stat, p_value = ks_2samp(a, b)
    return {
        "ks_statistic": float(stat),
        "ks_p_value": float(p_value),
    }


def js_divergence_score(a, b, grid_size=512, pad=0.1):
    """Jensen-Shannon divergence via KDE on a shared grid."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if a.size < 2 or b.size < 2:
        return np.nan
    if np.allclose(a.min(), a.max()) and np.allclose(b.min(), b.max()):
        return 0.0 if np.isclose(a[0], b[0]) else np.nan

    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    span = hi - lo
    lo -= pad * span
    hi += pad * span
    grid = np.linspace(lo, hi, grid_size)

    try:
        p = gaussian_kde(a)(grid)
        q = gaussian_kde(b)(grid)
    except (np.linalg.LinAlgError, ValueError):
        return np.nan

    p_sum, q_sum = p.sum(), q.sum()
    if p_sum == 0 or q_sum == 0:
        return np.nan
    p /= p_sum
    q /= q_sum

    js_distance = jensenshannon(p, q)
    return float(js_distance ** 2)


def _lehmer_l0_distribution(perms):
    if perms is None or len(perms) == 0:
        return None
    arr = np.vstack(perms)
    L = lehmer_encode_batch(arr)
    return L[:, 0].astype(float)

In [ ]:
def _normalize_keep_newlines(text):
    if text is None:
        return None
    s = str(text)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t\f\v]+", "", s)
    s = s.lower()
    return s


def _spaces_to_newlines(text):
    if text is None:
        return None
    s = str(text)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t\f\v]+", "\n", s)
    s = re.sub(r"\n+", "\n", s)
    return s


def _split_lines(text):
    if text is None:
        return None
    parts = [p for p in text.split("\n") if p != ""]
    return parts if parts else None


def _remove_word(text, word):
    if text is None:
        return None
    return re.sub(rf"\b{re.escape(word)}\b", "", text)


def _perm_list_to_indices(values, value_to_index):
    if values is None:
        return None
    indices = []
    for v in values:
        if v not in value_to_index:
            return None
        indices.append(value_to_index[v])
    if len(indices) != len(value_to_index):
        return None
    if len(set(indices)) != len(indices):
        return None
    return np.asarray(indices, dtype=int)

In [ ]:
# Load mapping config
with MAPPING_FILE.open("r", encoding="utf-8") as f:
    mapping_cfg = json.load(f)

def _uids_from_group(group):
    return [item["value"] for item in group.get("uids:", [])]

shuffling_uids = set(_uids_from_group(mapping_cfg.get("shuffling_separated_by_new_line", {})))
selection_uids = set(_uids_from_group(mapping_cfg.get("selection_separated_by_new_line", {})))
single_uids = set(_uids_from_group(mapping_cfg.get("single_item_selection", {})))

selection_mapping_by_uid = {}
for item in mapping_cfg.get("selection_separated_by_new_line", {}).get("uids:", []):
    uid = item.get("value")
    mapping = (item.get("metadata", {}) or {}).get("mapping")
    if mapping:
        normalized = {
            _normalize_keep_newlines(k): v for k, v in mapping.items()
        }
        selection_mapping_by_uid[uid] = normalized

single_mapping_by_uid = {}
single_type_by_uid = {}
for item in mapping_cfg.get("single_item_selection", {}).get("uids:", []):
    uid = item.get("value")
    metadata = item.get("metadata", {}) or {}
    single_type_by_uid[uid] = metadata.get("type")
    mapping = metadata.get("mapping")
    if mapping:
        normalized = {
            _normalize_keep_newlines(k): v for k, v in mapping.items()
        }
        single_mapping_by_uid[uid] = normalized

print("Shuffling UIDs:", len(shuffling_uids))
print("Selection UIDs:", len(selection_uids))
print("Single-item UIDs:", len(single_uids))

In [ ]:
# Load ground truth
with GT_FILE.open("r", encoding="utf-8") as f:
    gt_rows = json.load(f)

ground_truth_by_uid = {}
for row in gt_rows:
    uid = row.get("uid")
    values = row.get("ground_truth_values", [])
    if uid is None or not isinstance(values, list):
        continue
    ground_truth_by_uid[uid] = values

print(f"Loaded ground truth for {len(ground_truth_by_uid):,} UIDs.")

In [ ]:
# Build task dictionary keyed by uid
# Optional restriction to temp_1.0 and run_1..run_100

def _is_valid_run_path(path: Path) -> bool:
    parts = path.parts
    if "temp_1.0" not in parts:
        return False
    run_parts = [p for p in parts if p.startswith("run_")]
    if not run_parts:
        return False
    try:
        run_num = int(run_parts[-1].split("_", 1)[1])
    except (ValueError, IndexError):
        return False
    return 1 <= run_num <= 100

tasks = {}
all_result_files = sorted(RESULTS_ROOT.rglob("all_results.json"))
if USE_TEMP_1_RUN_1_100:
    result_files = [p for p in all_result_files if _is_valid_run_path(p)]
else:
    result_files = all_result_files

for result_file in result_files:
    with result_file.open("r", encoding="utf-8") as f:
        rows = json.load(f)

    for row in rows:
        uid = row.get("uid")
        if uid is None or uid not in ground_truth_by_uid:
            continue

        if uid not in tasks:
            tasks[uid] = {
                "uid": uid,
                "id": row.get("id"),
                "category": row.get("category"),
                "subcategory": row.get("subcategory"),
                "prompt_title": row.get("prompt_title"),
                "ground_truth_values": ground_truth_by_uid.get(uid, []),
                "model_outputs": defaultdict(list),
            }

        model_name = row.get("model_used")
        if model_name is None:
            continue

        if row.get("mapping_valid") is True and row.get("mapped_value") is not None:
            output_value = row.get("mapped_value")
        else:
            output_value = row.get("extracted_value")
            if row.get("mapping_valid") is False:
                if uid == "41b48a16-5c9a-5854-baad-20a4594e88ce":
                    output_value = "(no output)"
                else:
                    output_value = _spaces_to_newlines(output_value)

        tasks[uid]["model_outputs"][model_name].append(output_value)

for uid in tasks:
    tasks[uid]["model_outputs"] = dict(tasks[uid]["model_outputs"])

print(f"Loaded {len(result_files):,} result files.")
print(f"Built task dictionary for {len(tasks):,} UIDs.")

In [ ]:
RANDOM_MODEL_NAME = "random/gt-subsample"

def _compute_metrics_records(include_random_baseline=True):
    records = []

    for uid, task in tasks.items():
        gt_values = task.get("ground_truth_values", [])
        if not gt_values:
            continue

        if uid in shuffling_uids:
            normalized_gt = []
            for v in gt_values:
                s = _normalize_keep_newlines(v)
                s = _remove_word(s, "deleting") if uid == "c24f0c53-22dc-5a8d-a23a-874d3ac319ca" else s
                s = _remove_word(s, "replica") if uid == "e329c07f-2431-59a2-9cc1-4506484ee4f6" else s
                parts = _split_lines(s)
                if parts is None:
                    continue
                normalized_gt.append(parts)

            if not normalized_gt:
                continue

            value_to_index = {v: i for i, v in enumerate(normalized_gt[0])}
            if len(value_to_index) != len(normalized_gt[0]):
                continue

            gt_perms = []
            for perm in normalized_gt:
                perm_idx = _perm_list_to_indices(perm, value_to_index)
                if perm_idx is None:
                    continue
                gt_perms.append(perm_idx)

            if not gt_perms:
                continue

            gt_l0 = _lehmer_l0_distribution(gt_perms)
            if gt_l0 is None or gt_l0.size < 2:
                continue

            for model_name, outputs in task.get("model_outputs", {}).items():
                pred_perms = []
                for out in outputs:
                    s = _normalize_keep_newlines(out)
                    s = _remove_word(s, "deleting") if uid == "c24f0c53-22dc-5a8d-a23a-874d3ac319ca" else s
                    s = _remove_word(s, "replica") if uid == "e329c07f-2431-59a2-9cc1-4506484ee4f6" else s
                    parts = _split_lines(s)
                    if parts is None:
                        continue
                    perm_idx = _perm_list_to_indices(parts, value_to_index)
                    if perm_idx is None:
                        continue
                    pred_perms.append(perm_idx)

                pred_l0 = _lehmer_l0_distribution(pred_perms)
                if pred_l0 is None or pred_l0.size < 2:
                    continue

                ks = ks_score(gt_l0, pred_l0)
                rec = {
                    "uid": uid,
                    "prompt_title": task.get("prompt_title"),
                    "model": model_name,
                    "ks_statistic": ks["ks_statistic"],
                    "ks_p_value": ks["ks_p_value"],
                    "js_divergence": js_divergence_score(gt_l0, pred_l0),
                }

                for _n in [1, 2, 5, 10, 20, 50, 100]:
                    n_use = min(_n, pred_l0.size)
                    ks_at_n = ks_score(gt_l0, pred_l0[:n_use])
                    rec.update({
                        f"ks_{_n}_statistic": ks_at_n["ks_statistic"],
                        f"ks_{_n}_p_value": ks_at_n["ks_p_value"],
                    })
                records.append(rec)

            if include_random_baseline:
                n_max = min(100, gt_l0.size)
                random_pred = gt_l0[-n_max:]
                ks = ks_score(gt_l0, random_pred)
                rec = {
                    "uid": uid,
                    "prompt_title": task.get("prompt_title"),
                    "model": RANDOM_MODEL_NAME,
                    "ks_statistic": ks["ks_statistic"],
                    "ks_p_value": ks["ks_p_value"],
                    "js_divergence": js_divergence_score(gt_l0, random_pred),
                }
                for _n in [1, 2, 5, 10, 20, 50, 100]:
                    n_use = min(_n, random_pred.size)
                    ks_at_n = ks_score(gt_l0, random_pred[:n_use])
                    rec.update({
                        f"ks_{_n}_statistic": ks_at_n["ks_statistic"],
                        f"ks_{_n}_p_value": ks_at_n["ks_p_value"],
                    })
                records.append(rec)

        elif uid in selection_uids:
            mapping = selection_mapping_by_uid.get(uid)
            if not mapping:
                continue

            gt_mapped = []
            for v in gt_values:
                key = _normalize_keep_newlines(v)
                if key in mapping:
                    gt_mapped.append(mapping[key])
            if len(gt_mapped) < 2:
                continue
            gt_arr = np.asarray(gt_mapped, dtype=float)

            for model_name, outputs in task.get("model_outputs", {}).items():
                pred_mapped = []
                for out in outputs:
                    key = _normalize_keep_newlines(out)
                    if key in mapping:
                        pred_mapped.append(mapping[key])
                if len(pred_mapped) < 2:
                    continue
                pred_arr = np.asarray(pred_mapped, dtype=float)
                ks = ks_score(gt_arr, pred_arr)
                rec = {
                    "uid": uid,
                    "prompt_title": task.get("prompt_title"),
                    "model": model_name,
                    "ks_statistic": ks["ks_statistic"],
                    "ks_p_value": ks["ks_p_value"],
                    "js_divergence": js_divergence_score(gt_arr, pred_arr),
                }
                for _n in [1, 2, 5, 10, 20, 50, 100]:
                    n_use = min(_n, pred_arr.size)
                    ks_at_n = ks_score(gt_arr, pred_arr[:n_use])
                    rec.update({
                        f"ks_{_n}_statistic": ks_at_n["ks_statistic"],
                        f"ks_{_n}_p_value": ks_at_n["ks_p_value"],
                    })
                records.append(rec)

            if include_random_baseline:
                n_max = min(100, gt_arr.size)
                random_pred = gt_arr[-n_max:]
                ks = ks_score(gt_arr, random_pred)
                rec = {
                    "uid": uid,
                    "prompt_title": task.get("prompt_title"),
                    "model": RANDOM_MODEL_NAME,
                    "ks_statistic": ks["ks_statistic"],
                    "ks_p_value": ks["ks_p_value"],
                    "js_divergence": js_divergence_score(gt_arr, random_pred),
                }
                for _n in [1, 2, 5, 10, 20, 50, 100]:
                    n_use = min(_n, random_pred.size)
                    ks_at_n = ks_score(gt_arr, random_pred[:n_use])
                    rec.update({
                        f"ks_{_n}_statistic": ks_at_n["ks_statistic"],
                        f"ks_{_n}_p_value": ks_at_n["ks_p_value"],
                    })
                records.append(rec)

        elif uid in single_uids:
            mapping = single_mapping_by_uid.get(uid)
            gt_mapped = []
            if mapping:
                for v in gt_values:
                    key = _normalize_keep_newlines(v)
                    if key in mapping:
                        gt_mapped.append(mapping[key])
            else:
                for v in gt_values:
                    try:
                        gt_mapped.append(float(v))
                    except (TypeError, ValueError):
                        continue

            if len(gt_mapped) < 2:
                continue
            gt_arr = np.asarray(gt_mapped, dtype=float)

            for model_name, outputs in task.get("model_outputs", {}).items():
                pred_mapped = []
                if mapping:
                    for out in outputs:
                        key = _normalize_keep_newlines(out)
                        if key in mapping:
                            pred_mapped.append(mapping[key])
                else:
                    for out in outputs:
                        try:
                            pred_mapped.append(float(out))
                        except (TypeError, ValueError):
                            continue

                if len(pred_mapped) < 2:
                    continue
                pred_arr = np.asarray(pred_mapped, dtype=float)
                ks = ks_score(gt_arr, pred_arr)
                rec = {
                    "uid": uid,
                    "prompt_title": task.get("prompt_title"),
                    "model": model_name,
                    "ks_statistic": ks["ks_statistic"],
                    "ks_p_value": ks["ks_p_value"],
                    "js_divergence": js_divergence_score(gt_arr, pred_arr),
                }
                for _n in [1, 2, 5, 10, 20, 50, 100]:
                    n_use = min(_n, pred_arr.size)
                    ks_at_n = ks_score(gt_arr, pred_arr[:n_use])
                    rec.update({
                        f"ks_{_n}_statistic": ks_at_n["ks_statistic"],
                        f"ks_{_n}_p_value": ks_at_n["ks_p_value"],
                    })
                records.append(rec)

            if include_random_baseline:
                n_max = min(100, gt_arr.size)
                random_pred = gt_arr[-n_max:]
                ks = ks_score(gt_arr, random_pred)
                rec = {
                    "uid": uid,
                    "prompt_title": task.get("prompt_title"),
                    "model": RANDOM_MODEL_NAME,
                    "ks_statistic": ks["ks_statistic"],
                    "ks_p_value": ks["ks_p_value"],
                    "js_divergence": js_divergence_score(gt_arr, random_pred),
                }
                for _n in [1, 2, 5, 10, 20, 50, 100]:
                    n_use = min(_n, random_pred.size)
                    ks_at_n = ks_score(gt_arr, random_pred[:n_use])
                    rec.update({
                        f"ks_{_n}_statistic": ks_at_n["ks_statistic"],
                        f"ks_{_n}_p_value": ks_at_n["ks_p_value"],
                    })
                records.append(rec)

    return records

metrics_df_full = pd.DataFrame(_compute_metrics_records(include_random_baseline=True))
print(f"Computed rows: {len(metrics_df_full):,}")
metrics_df_full.head()

In [ ]:
# Accuracy table from KS p-values
threshold = 0.0001

metrics_df_full["acc_ks"] = metrics_df_full["ks_p_value"] > threshold

for _n in [1, 2, 5, 10, 20, 50, 100]:
    metrics_df_full[f"acc_ks_{_n}"] = metrics_df_full[f"ks_{_n}_p_value"] > threshold

df_accs = metrics_df_full[[
    "acc_ks", "acc_ks_1", "acc_ks_2", "acc_ks_5",
    "acc_ks_10", "acc_ks_20", "acc_ks_50", "acc_ks_100", "model",
]].groupby("model").mean()

sorted_df_accs = df_accs.sort_values("acc_ks", ascending=False)
sorted_df_accs